In [ ]:
!pip install MEDS-Inspect

In [ ]:
MEDS_Inspect_cache "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

In [ ]:
!MEDS_Inspect port=8052 +initial_path="/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

# Imports

In [ ]:
import os
import pandas as pd
import subprocess
import numpy as np
import hail as hl
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from datetime import datetime

In [ ]:
import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"

In [ ]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 200g "
    "--conf spark.driver.maxResultSize=16g "
    "--conf spark.default.parallelism=64 "
    "--conf spark.sql.shuffle.partitions=256 "
    "pyspark-shell"
)

import hail as hl

hl.init(
    master="local[32]",
    idempotent=True,
    default_reference = "GRCh38"
)

In [ ]:
hl.stop()
hl.init(default_reference = "GRCh38")

# Clinical Data

In [ ]:
dataset_08947253_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

dataset_08947253_person_df = pd.read_gbq(
    dataset_08947253_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08947253_person_df.head(5)

In [ ]:
DATA_BUCKET = '/home/jupyter/workspace/data_bucket'
GENETIC_FOLDER = f'{DATA_BUCKET}/v9_gen_data'
dataset_08947253_person_df.to_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t", index=False)
dataset_hl = (hl.import_table(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv",
                              types={'person_id':hl.tstr},
                              impute=True,
                              key='person_id')
             )

In [ ]:
dataset_hl.summarize()

# Genetic Data

In [ ]:
import hail as hl
import pandas as pd

vat_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/aux/vat/vat_complete.bgz.tsv.gz"
)

vat_table = hl.import_table(
    vat_path,
    force=True,
    quote='"',
    delimiter="\t",
    force_bgz=True,
    types={
        "position": hl.tint32,
        "contig": hl.tstr,
        "ref_allele": hl.tstr,
        "alt_allele": hl.tstr,
    },
)

vat_table.describe()

In [ ]:
transcripts_of_interest = [
    "ENST00000279146", "ENST00000389048", "ENST00000257430", "ENST00000675843", "ENST00000307078",
    "ENST00000460680", "ENST00000260947", "ENST00000355112", "ENST00000372037", "ENST00000357654",
    "ENST00000380152", "ENST00000259008", "ENST00000639785", "ENST00000367435", "ENST00000261769",
    "ENST00000257904", "ENST00000228872", "ENST00000313407", "ENST00000440480", "ENST00000304494",
    "ENST00000579755", "ENST00000498907", "ENST00000404276", "ENST00000302763", "ENST00000343455",
    "ENST00000325385", "ENST00000275493", "ENST00000263735", "ENST00000366560", "ENST00000285071",
    "ENST00000341105", "ENST00000487848", "ENST00000370818", "ENST00000651081", "ENST00000300177",
    "ENST00000651154", "ENST00000290295", "ENST00000610977", "ENST00000311189", "ENST00000417302",
    "ENST00000288135", "ENST00000358664", "ENST00000555147", "ENST00000450708", "ENST00000397752",
    "ENST00000394351", "ENST00000352241", "ENST00000231790", "ENST00000233146", "ENST00000265081",
    "ENST00000234420", "ENST00000456914", "ENST00000710952", "ENST00000265433", "ENST00000358273",
    "ENST00000338641", "ENST00000651570", "ENST00000261584", "ENST00000257290", "ENST00000226382",
    "ENST00000265849", "ENST00000440232", "ENST00000320574", "ENST00000357628", "ENST00000589228",
    "ENST00000331920", "ENST00000437951", "ENST00000644628", "ENST00000371953", "ENST00000378823",
    "ENST00000337432", "ENST00000345365", "ENST00000267163", "ENST00000617875", "ENST00000355710",
    "ENST00000675419", "ENST00000264932", "ENST00000301761", "ENST00000375499", "ENST00000367975",
    "ENST00000375549", "ENST00000342988", "ENST00000344626", "ENST00000646693", "ENST00000618915",
    "ENST00000644036", "ENST00000348513", "ENST00000326873", "ENST00000369902", "ENST00000310581",
    "ENST00000258439", "ENST00000269305", "ENST00000298552", "ENST00000219476", "ENST00000256474",
    "ENST00000298139", "ENST00000452863"
]

transcripts_literal = hl.literal(set(transcripts_of_interest))

transcript_vat_table = vat_table.annotate(
    transcript_id=hl.or_else(
        vat_table.transcript,
        ""
    ).split(r"\.")[0]
)

transcript_vat_table = transcript_vat_table.filter(
    transcripts_literal.contains(
        transcript_vat_table.transcript_id
    )
)

transcript_vat_table.describe()

In [ ]:
accepted_classifications = hl.literal({
    "pathogenic",
    "likely pathogenic",
    "likely risk allele",
    "risk factor",
})

filtered_vat_table = transcript_vat_table.annotate(
    classification_terms=(
        hl.or_else(
            transcript_vat_table.clinvar_classification,
            ""
        )
        .lower()
        .split(",")
        .map(lambda value: value.strip())
    )
)

filtered_vat_table = filtered_vat_table.filter(
    hl.any(
        lambda classification: accepted_classifications.contains(
            classification
        ),
        filtered_vat_table.classification_terms,
    )
)

filtered_vat_table = filtered_vat_table.drop(
    "classification_terms"
)

filtered_vat_table = filtered_vat_table.annotate(
    locus=hl.locus(
        filtered_vat_table.contig,
        filtered_vat_table.position,
        reference_genome="GRCh38",
    ),
    alleles=[
        filtered_vat_table.ref_allele,
        filtered_vat_table.alt_allele,
    ],
)

excluded_consequences = hl.literal({
    "downstream_gene_variant",
    "upstream_gene_variant",
})

filtered_vat_table = filtered_vat_table.filter(
    hl.is_missing(filtered_vat_table.consequence)
    | ~excluded_consequences.contains(
        filtered_vat_table.consequence
    )
)

filtered_vat_table = filtered_vat_table.key_by(
    "locus",
    "alleles",
)

filtered_vat_by_variant = (
    filtered_vat_table
    .group_by(
        locus=filtered_vat_table.locus,
        alleles=filtered_vat_table.alleles,
    )
    .aggregate(
        annotations=hl.agg.collect(
            filtered_vat_table.row_value
        ),

        gene_symbols=hl.agg.filter(
            hl.is_defined(filtered_vat_table.gene_symbol)
            & (filtered_vat_table.gene_symbol != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.gene_symbol
            ),
        ),

        transcripts=hl.agg.filter(
            hl.is_defined(filtered_vat_table.transcript)
            & (filtered_vat_table.transcript != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.transcript
            ),
        ),

        consequences=hl.agg.filter(
            hl.is_defined(filtered_vat_table.consequence)
            & (filtered_vat_table.consequence != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.consequence
            ),
        ),

        clinvar_classifications=hl.agg.filter(
            hl.is_defined(
                filtered_vat_table.clinvar_classification
            ),
            hl.agg.collect_as_set(
                filtered_vat_table.clinvar_classification
            ),
        ),
    )
)

filtered_vat_by_variant.describe()

In [ ]:
df = pd.read_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t")
df_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/df_whole_DS_v8.tsv', sep='t')

In [ ]:
meta_v9 = pd.read_csv(f"{GENETIC_FOLDER}/metadata_v9.csv")
meta_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/metadata_v8.csv')

In [ ]:
samples_v8 = set(meta_v8["s"].dropna().unique())
samples_v9 = set(meta_v9["s"].dropna().unique())


print("v8 total:", len(samples_v8))
print("v9 total:", len(samples_v9))
print("v8 also in v9:", len(samples_v8 & samples_v9))
print("v8 missing from v9:", len(samples_v8 - samples_v9))
print("new in v9:", len(samples_v9 - samples_v8))

In [ ]:
meta_v9[meta_v9['s']==int(list(samples_v8 & samples_v9 & samples_v8_filt & (samples_v8_filt - samples_v9_filt))[11])]

In [ ]:
meta_v8[meta_v8['s']==int(list(samples_v8 & samples_v9 & samples_v8_filt & (samples_v8_filt - samples_v9_filt))[11])]

In [ ]:
mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)
mt_9 = hl.read_matrix_table(mt_wgs_clinvar_path)

In [ ]:
mt_9.filter_cols(mt_9.s == str(1736721))

In [ ]:
entries_v8[entries_v8.s==1736721]

In [ ]:
patient_id = "1736721"

target_locus = hl.parse_locus(
    "chr1:45331556",
    reference_genome="GRCh38"
)

result_mt = mt_9.filter_cols(mt_9.s == patient_id)

result_mt = result_mt.filter_rows(
    (result_mt.locus == target_locus) &
    (result_mt.alleles == ["C", "T"])
)

result_entries = result_mt.entries()

result_entries.show(n=100, width=200)

In [ ]:
!pip install pysam
import pysam
import pandas as pd

vat_file = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/aux/vat/vat_complete.bgz.tsv.gz"
)

tbi_file = vat_file + ".tbi"

# Read the actual first line of the compressed VAT
with pysam.BGZFile(vat_file, "r") as f:
    header_line = f.readline().decode("utf-8").rstrip("\n")

columns = header_line.lstrip("#").split("\t")

print(columns)
tbx = pysam.TabixFile(vat_file, index=tbi_file)

rows = tbx.fetch(
    "chr1",
    45331555,  # zero-based start
    45331556   # exclusive end
)

contig_i = columns.index("contig")
position_i = columns.index("position")
ref_i = columns.index("ref_allele")
alt_i = columns.index("alt_allele")

matches = []

for row in rows:
    fields = row.rstrip("\n").split("\t")

    if (
        fields[contig_i] == "chr1"
        and int(fields[position_i]) == 45331556
        and fields[ref_i] == "C"
        and fields[alt_i] == "T"
    ):
        matches.append(dict(zip(columns, fields)))

variant_df = pd.DataFrame(matches)
variant_df.clinvar_classification

In [ ]:
sample_id = str(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])

target_locus = hl.locus("17", 43071077, reference_genome="GRCh37")

individual_mt = mt_9.filter_cols(mt_9.s == sample_id)

individual_mt.entries().show()
target_locus = hl.parse_locus(
    "chr1:45331556",
    reference_genome="GRCh38"
)

result = mt_9.filter_rows(mt_9.locus == target_locus)
result = result.filter_cols(result.s == sample_id)

genes_of_interest = [
    "AIP", "ALK", "APC", "ATM", "AXIN2", "BAP1", "BARD1", "BLM",
    "BMPR1A", "BRCA1", "BRCA2", "BRIP1", "CASR", "CDC73", "CDH1",
    "CDK4", "CDKN1B", "CDKN1C", "CDKN2A", "CEBPA", "CHEK2",
    "CTNNA1", "DICER1", "DIS3L2", "EGFR", "EPCAM", "FH", "FLCN",
    "GATA2", "GPC3", "GREM1", "HOXB13", "HRAS", "KIT", "MAX",
    "MC1R", "MEN1", "MET", "MITF", "MLH1", "MSH2", "MSH3", "MSH6",
    "MUTYH", "NBN", "NF1", "NF2", "NTHL1", "PALB2", "PDGFRA",
    "PHOX2B", "PMS2", "POLD1", "POLE", "POT1", "PRKAR1A", "PTCH1",
    "PTEN", "RAD50", "RAD51C", "RAD51D", "RB1", "RECQL4", "RET",
    "RUNX1", "SDHA", "SDHAF2", "SDHB", "SDHC", "SDHD", "SMAD4",
    "SMARCA4", "SMARCB1", "SMARCE1", "STK11", "SUFU", "TERC", "TERT",
    "TMEM127", "TP53", "TSC1", "TSC2", "VHL", "WRN", "WT1"
]

dataset_hl[result.s].show()

print(hl.grep(
    r"^chr1\t45331556\tC\tT\t",
    vat_path,
    max_count=1
))

# result.entries().show()

# # Keep only samples represented in dataset_hl.
# mt_sub = result.semi_join_cols(dataset_hl)

# # Keep only variants represented in filtered_vat_table.
# mt_sub = mt_sub.semi_join_rows(filtered_vat_table)

# mt_sub.describe()

# # Potentially expensive.
# # print(mt_sub.count())


# # ---------------------------------------------------------------------
# # Add patient metadata and complete VAT annotations
# # ---------------------------------------------------------------------

# mt_sub = mt_sub.annotate_cols(
#     metadata=dataset_hl[mt_sub.s]
# )

# mt_sub = mt_sub.annotate_rows(
#     annotations=filtered_vat_table[
#         mt_sub.locus,
#         mt_sub.alleles
#     ]
# )


# genes_literal = hl.literal(set(genes_of_interest))

# mt_filtered = mt_sub.filter_rows(
#     hl.is_defined(mt_sub.annotations.gene_symbol)
#     & genes_literal.contains(mt_sub.annotations.gene_symbol)
# )


# # ---------------------------------------------------------------------
# # Identify non-reference genotypes
# # ---------------------------------------------------------------------

# mt_filtered = mt_filtered.annotate_entries(
#     has_variant=(
#         hl.is_defined(mt_filtered.GT)
#         & mt_filtered.GT.is_non_ref()
#     )
# )

# mt_filtered.describe()


# # ---------------------------------------------------------------------
# # Create entries table and keep non-reference genotypes
# # ---------------------------------------------------------------------

# entries_table = mt_filtered.entries()

# entries_table = entries_table.filter(
#     entries_table.has_variant
# )

# entries_table.to_pandas()


In [ ]:
pd.set_option("display.max_columns", None)
entries_v8[entries_v8.s == int(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])]

In [ ]:
meta_v9[meta_v9.s==int(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])]

In [ ]:
import csv
import gzip
import os
import shutil
from pathlib import Path

import hail as hl


# ============================================================
# Configuration
# ============================================================

mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)

genetic_folder = Path(GENETIC_FOLDER).expanduser().resolve()
genetic_folder.mkdir(parents=True, exist_ok=True)

metadata_output_path = genetic_folder / "metadata_v9.csv"
entries_output_path = genetic_folder / "entries_table_full_v9.csv"

# This folder contains resumable Hail checkpoints and temporary exports.
# It is intentionally versioned so it does not reuse checkpoints produced
# by the broken pipeline.
work_folder = genetic_folder / "_hail_v9_export_v2"
work_folder.mkdir(parents=True, exist_ok=True)

joined_mt_path = work_folder / "joined_annotated.mt"
carrier_mt_path = work_folder / "carrier_only.mt"
metadata_ht_path = work_folder / "metadata.ht"
entries_ht_path = work_folder / "entries.ht"

metadata_tsv_path = work_folder / "metadata_v9.tsv.bgz"
entries_tsv_path = work_folder / "entries_table_full_v9.tsv.bgz"


genes_of_interest = [
    "AIP", "ALK", "APC", "ATM", "AXIN2", "BAP1", "BARD1", "BLM",
    "BMPR1A", "BRCA1", "BRCA2", "BRIP1", "CASR", "CDC73", "CDH1",
    "CDK4", "CDKN1B", "CDKN1C", "CDKN2A", "CEBPA", "CHEK2",
    "CTNNA1", "DICER1", "DIS3L2", "EGFR", "EPCAM", "FH", "FLCN",
    "GATA2", "GPC3", "GREM1", "HOXB13", "HRAS", "KIT", "MAX",
    "MC1R", "MEN1", "MET", "MITF", "MLH1", "MSH2", "MSH3", "MSH6",
    "MUTYH", "NBN", "NF1", "NF2", "NTHL1", "PALB2", "PDGFRA",
    "PHOX2B", "PMS2", "POLD1", "POLE", "POT1", "PRKAR1A", "PTCH1",
    "PTEN", "RAD50", "RAD51C", "RAD51D", "RB1", "RECQL4", "RET",
    "RUNX1", "SDHA", "SDHAF2", "SDHB", "SDHC", "SDHD", "SMAD4",
    "SMARCA4", "SMARCB1", "SMARCE1", "STK11", "SUFU", "TERC", "TERT",
    "TMEM127", "TP53", "TSC1", "TSC2", "VHL", "WRN", "WT1",
]

genes_literal = hl.literal(set(genes_of_interest))


# ============================================================
# Utility functions
# ============================================================

def hail_uri(path: Path) -> str:
    """Convert an absolute local Path to a file:// URI for Hail."""
    return path.resolve().as_uri()


def delete_path(path: Path) -> None:
    """Delete a file or directory left by an interrupted operation."""
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()


def read_or_build_mt(path: Path, builder):
    """
    Read a completed MatrixTable checkpoint.

    If it does not exist or is corrupted/incomplete, rebuild it.
    """
    uri = hail_uri(path)

    if path.exists():
        try:
            print(f"Reading MatrixTable checkpoint: {path}")
            return hl.read_matrix_table(uri)
        except Exception as exc:
            print(f"Invalid checkpoint; rebuilding {path}")
            print(f"Checkpoint read error: {exc}")
            delete_path(path)

    print(f"Building MatrixTable checkpoint: {path}")
    mt = builder()

    # checkpoint() writes and immediately reads the stored result,
    # cutting off the previous computation graph.
    return mt.checkpoint(
        uri,
        overwrite=True,
    )


def read_or_build_ht(path: Path, builder):
    """
    Read a completed Table checkpoint.

    If it does not exist or is corrupted/incomplete, rebuild it.
    """
    uri = hail_uri(path)

    if path.exists():
        try:
            print(f"Reading Table checkpoint: {path}")
            return hl.read_table(uri)
        except Exception as exc:
            print(f"Invalid checkpoint; rebuilding {path}")
            print(f"Checkpoint read error: {exc}")
            delete_path(path)

    print(f"Building Table checkpoint: {path}")
    ht = builder()

    return ht.checkpoint(
        uri,
        overwrite=True,
    )


def export_hail_tsv(ht: hl.Table, output_path: Path) -> None:
    """
    Export a Hail table without collecting it into Python.

    A completion marker distinguishes a valid completed export from a
    partial file left behind by an interrupted notebook.
    """
    done_path = Path(f"{output_path}.done")

    if output_path.exists() and done_path.exists():
        print(f"Using completed Hail export: {output_path}")
        return

    delete_path(output_path)
    delete_path(done_path)

    print(f"Exporting Hail table to: {output_path}")

    # parallel=None produces one file.
    # .bgz keeps the temporary export substantially smaller.
    ht.export(
        hail_uri(output_path),
        delimiter="\t",
        header=True,
        parallel=None,
    )

    done_path.touch()

    print(f"Completed Hail export: {output_path}")


def convert_bgz_tsv_to_csv(
    input_path: Path,
    output_path: Path,
) -> int:
    """
    Convert BGZF/TSV to CSV one row at a time.

    This uses constant memory and correctly quotes commas contained in
    ClinVar classifications, JSON structs, and other fields.
    """
    done_path = Path(f"{output_path}.done")
    partial_path = Path(f"{output_path}.partial")

    if output_path.exists() and done_path.exists():
        try:
            row_count = int(done_path.read_text().strip())
        except Exception:
            row_count = -1

        print(
            f"Using completed CSV: {output_path} "
            f"({row_count:,} data rows)"
        )
        return row_count

    delete_path(partial_path)
    delete_path(done_path)

    print(f"Converting to CSV: {output_path}")

    row_count = 0

    with gzip.open(
        input_path,
        mode="rt",
        encoding="utf-8",
        newline="",
    ) as input_file, open(
        partial_path,
        mode="w",
        encoding="utf-8",
        newline="",
    ) as output_file:

        reader = csv.reader(
            input_file,
            delimiter="\t",
        )

        writer = csv.writer(
            output_file,
            quoting=csv.QUOTE_MINIMAL,
        )

        try:
            header = next(reader)
        except StopIteration as exc:
            raise RuntimeError(
                f"Hail produced an empty export with no header: "
                f"{input_path}"
            ) from exc

        writer.writerow(header)

        for row in reader:
            writer.writerow(row)
            row_count += 1

    # Atomic replacement: entries_table_full_v9.csv appears only after
    # the conversion has completed successfully.
    os.replace(
        partial_path,
        output_path,
    )

    done_path.write_text(str(row_count))

    print(
        f"Completed CSV: {output_path} "
        f"({row_count:,} data rows)"
    )

    return row_count


# ============================================================
# Stage 1: Filter and annotate the source MatrixTable
# ============================================================

def build_joined_mt():
    print("Reading ClinVar MatrixTable")

    mt = hl.read_matrix_table(
        mt_wgs_clinvar_path
    )

    print("Restricting to selected participants")

    mt = mt.semi_join_cols(
        dataset_hl
    )

    print("Restricting to selected VAT variants")

    mt = mt.semi_join_rows(
        filtered_vat_by_variant
    )

    print("Adding participant metadata")

    mt = mt.annotate_cols(
        metadata=dataset_hl[mt.s]
    )

    print("Adding VAT annotations")

    vat = filtered_vat_by_variant[
        mt.locus,
        mt.alleles,
    ]

    mt = mt.annotate_rows(
        annotations=vat.annotations,
        gene_symbols=vat.gene_symbols,
        transcripts=vat.transcripts,
        consequences=vat.consequences,
        clinvar_classifications=(
            vat.clinvar_classifications
        ),
    )

    return mt


joined_mt = read_or_build_mt(
    joined_mt_path,
    build_joined_mt,
)


# ============================================================
# Stage 2: Export participant metadata separately
# ============================================================

def build_metadata_ht():
    # One row per participant. Metadata is not repeated for every
    # person-variant entry.
    return joined_mt.cols()


metadata_ht = read_or_build_ht(
    metadata_ht_path,
    build_metadata_ht,
)

export_hail_tsv(
    metadata_ht,
    metadata_tsv_path,
)

metadata_count = convert_bgz_tsv_to_csv(
    metadata_tsv_path,
    metadata_output_path,
)


# ============================================================
# Stage 3: Keep requested genes and NON-REFERENCE ENTRIES ONLY
# ============================================================

def build_carrier_mt():
    print("Filtering to genes of interest")

    mt = joined_mt.filter_rows(
        hl.is_defined(joined_mt.gene_symbols)
        & hl.any(
            lambda gene: genes_literal.contains(gene),
            joined_mt.gene_symbols,
        )
    )

    if "GT" not in mt.entry.dtype.fields:
        raise RuntimeError(
            "The ClinVar MatrixTable does not contain an entry field "
            "named GT."
        )

    print("Filtering entries to non-reference genotypes")

    # This is the critical change.
    #
    # The non-carrier entries are removed from the MatrixTable before
    # entries() is called. They will therefore never appear in the
    # coordinate table.
    mt = mt.filter_entries(
        hl.is_defined(mt.GT)
        & mt.GT.is_non_ref()
    )

    # Remove variants that have no carriers in the selected cohort.
    mt = mt.filter_rows(
        hl.agg.count() > 0
    )

    # Keep only the genotype fields actually needed in the CSV.
    # Add another field to this list only when downstream analysis uses it.
    preferred_entry_fields = [
        "GT",
        "AD",
        "DP",
        "GQ",
        "PL",
    ]

    retained_entry_fields = [
        field
        for field in preferred_entry_fields
        if field in mt.entry.dtype.fields
    ]

    print(
        "Entry fields retained:",
        retained_entry_fields,
    )

    mt = mt.select_entries(
        *retained_entry_fields
    )

    # Keep the variant annotations that the output needs.
    mt = mt.select_rows(
        "annotations",
        "gene_symbols",
        "transcripts",
        "consequences",
        "clinvar_classifications",
    )

    # Drop the metadata struct from the entry table. It is already stored
    # once per person in metadata_v9.csv. Keeping it here would duplicate
    # the entire metadata record for every variant carried by that person.
    mt = mt.select_cols()

    return mt


carrier_mt = read_or_build_mt(
    carrier_mt_path,
    build_carrier_mt,
)


# ============================================================
# Stage 4: Produce the sparse person-variant entries table
# ============================================================

def build_entries_ht():
    # Unkeying the columns prevents Hail from unnecessarily sorting the
    # output by the compound (variant, sample) key.
    mt = carrier_mt.key_cols_by()

    ht = mt.entries()

    # The output does not require a Hail table key.
    return ht.key_by()


entries_ht = read_or_build_ht(
    entries_ht_path,
    build_entries_ht,
)


# ============================================================
# Stage 5: Export without pandas
# ============================================================

export_hail_tsv(
    entries_ht,
    entries_tsv_path,
)

entries_count = convert_bgz_tsv_to_csv(
    entries_tsv_path,
    entries_output_path,
)


# ============================================================
# Final verification
# ============================================================

if not metadata_output_path.exists():
    raise RuntimeError(
        f"Metadata CSV was not created: {metadata_output_path}"
    )

if not entries_output_path.exists():
    raise RuntimeError(
        f"Entries CSV was not created: {entries_output_path}"
    )

print()
print("=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)
print(
    f"Metadata: {metadata_output_path} "
    f"({metadata_count:,} rows)"
)
print(
    f"Entries:  {entries_output_path} "
    f"({entries_count:,} rows)"
)
print(
    f"Metadata size: "
    f"{metadata_output_path.stat().st_size / (1024 ** 3):.3f} GiB"
)
print(
    f"Entries size:  "
    f"{entries_output_path.stat().st_size / (1024 ** 3):.3f} GiB"
)

### Deprecated

In [ ]:
print('hi')

In [ ]:
# Load the existing filtered MatrixTable
mt = hl.read_matrix_table(f"{GENETIC_FOLDER}/filtered_nonref.mt")

# Restore all VAT annotation fields
mt = mt.annotate_rows(
    annotations=filtered_vat_table[mt.locus, mt.alleles]
)

# Restore the V8 column, if needed
mt = mt.annotate_entries(
    has_variant=hl.is_defined(mt.GT) & mt.GT.is_non_ref()
)

# Recreate only the entries table
entries_table = mt.entries()

# Convert and save
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(
    f"{GENETIC_FOLDER}/entries_table_full_v9.csv",
    index=False
)

print(entries_table_df.shape)

In [ ]:
mt = hl.read_matrix_table(filtered_mt_path)
# Extract only needed fields
entries_table = mt.entries()

# Write distributed Hail Table first
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"

entries_table = entries_table.checkpoint(
    entries_ht_path,
    overwrite=True
)

# Export as multiple compressed TSV shards
entries_table.export(
    f"{GENETIC_FOLDER}/entries_table_v9.tsv.bgz",
    parallel="header_per_shard"
)

In [ ]:
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"
entries_table = hl.read_table(entries_ht_path)
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(f'{GENETIC_FOLDER}/entries_table_full_v9.csv', index=False)

In [ ]:
entries = pd.read_csv(f"{GENETIC_FOLDER}/entries_table_full_v9.csv")

In [ ]:
entries[entries['gene_symbol']=='SDHB']

In [ ]:
mt_wgs_clinvar_path = '/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/clinvar/splitMT/hail.mt'

mt = hl.read_matrix_table(mt_wgs_clinvar_path)
mt.describe()

In [ ]:
# Select only the samples in dataset_hl
mt_sub = mt.semi_join_cols(dataset_hl)
# Select only the variants in filtered_vat_table
mt_sub = mt_sub.semi_join_rows(filtered_vat_table)
mt_sub.describe()

# Annotate MatrixTable columns with metadata
mt_sub = mt_sub.annotate_cols(metadata=dataset_hl[mt_sub.s])
# Annotate MatrixTable rows with variant annotations
mt_sub = mt_sub.annotate_rows(annotations=filtered_vat_table[mt_sub.locus, mt_sub.alleles])
# Extract the column fields (metadata and sample ID 's') into a Hail Table
metadata_table = mt_sub.cols()

# Convert the Hail Table to a Pandas DataFrame
metadata_df = metadata_table.to_pandas()

# Save the DataFrame to a CSV file
metadata_df.to_csv(f'{GENETIC_FOLDER}/metadata_v9.csv', index=False)

In [ ]:
# Kind of redundant though
# Filter the MatrixTable to specific genes
genes_of_interest = ['AIP', 'ALK', 'APC', 'ATM', 'AXIN2', 'BAP1', 'BARD1', 'BLM', 'BMPR1A', 'BRCA1', 'BRCA2', 
                     'BRIP1', 'CASR', 'CDC73', 'CDH1', 'CDK4', 'CDKN1B', 'CDKN1C', 'CDKN2A', 'CEBPA', 'CHEK2', 
                     'CTNNA1', 'DICER1', 'DIS3L2', 'EGFR', 'EPCAM', 'FH', 'FLCN', 'GATA2', 'GPC3', 'GREM1', 
                     'HOXB13', 'HRAS', 'KIT', 'MAX', 'MC1R', 'MEN1', 'MET', 'MITF', 'MLH1', 'MSH2', 'MSH3', 
                     'MSH6', 'MUTYH', 'NBN', 'NF1', 'NF2', 'NTHL1', 'PALB2', 'PDGFRA', 'PHOX2B', 'PMS2', 
                     'POLD1', 'POLE', 'POT1', 'PRKAR1A', 'PTCH1', 'PTEN', 'RAD50', 'RAD51C', 'RAD51D', 'RB1', 
                     'RECQL4', 'RET', 'RUNX1', 'SDHA', 'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'SMARCA4', 
                     'SMARCB1', 'SMARCE1', 'STK11', 'SUFU', 'TERC', 'TERT', 'TMEM127', 'TP53', 'TSC1', 'TSC2', 
                     'VHL', 'WRN', 'WT1']

mt_filtered = mt_sub.filter_rows(hl.literal(genes_of_interest).contains(mt_sub.annotations.gene_symbol))
mt_filtered = mt_filtered.annotate_entries(has_variant=mt_filtered.GT.is_non_ref())
mt_filtered.describe()

In [ ]:
entries_table = mt_filtered.entries()
# Filter entries to include only non-reference variants
entries_table = entries_table.filter(entries_table.has_variant)
entries_table.describe()

# Analysis

In [ ]:
entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
entries_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
samples_v8_filt = set(entries_v8["s"].dropna().unique())
samples_v9_filt = set(entries_v9["s"].dropna().unique())


print("v8 total:", len(samples_v8_filt))
print("v9 total:", len(samples_v9_filt))
print("v8 also in v9:", len(samples_v8_filt & samples_v9_filt))
print("v8 missing from v9:", len(samples_v8_filt - samples_v9_filt))
print("new in v9:", len(samples_v9_filt - samples_v8_filt))

In [ ]:
print(entries_v9.s.nunique())
print(entries_v8.s.nunique())

In [ ]:
entries_table_df = entries_v9[entries_v9['annotations.gene_symbol'] != 'MC1R']

entries_table_df = entries_table_df[
    ~(
        ((entries_table_df['annotations.gene_symbol'] == 'EPCAM') & 
         (entries_table_df['annotations.variant_type'] == 'deletion')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'PDGFRA') & 
         (entries_table_df['annotations.vid'] == '4-54281602-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'EGFR') & 
         (entries_table_df['annotations.vid'] == '7-55173126-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43100576-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43106497-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TSC1') & 
         (entries_table_df['annotations.vid'] == '9-132921940-T-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TMEM127') & 
         (entries_table_df['annotations.vid'] == '2-96265399-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1293489-C-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1268581-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1254461-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132680048-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132677577-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'ALK') & 
         (entries_table_df['annotations.vid'] == '2-29220747-C-T'))
    )
]



In [ ]:
entries_table_df.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v8.csv')